In [3]:
!pip install lightly #good library for contrastive learning based on google search
!pip install scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 50.8 MB/s eta 0:00:00
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn]━━━━━ 2/3 [scikit-learn]


In [4]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from lightly.models.modules import heads
from lightly.loss import NTXentLoss
from lightly.transforms import SimCLRTransform
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from cub2011 import Cub2011
import torchvision.models as models
import random
from tqdm import tqdm
import sklearn.preprocessing
import logging
import sklearn.cluster
import sklearn.metrics.cluster

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Transform pipeline
Apply random crop, horizontal flip, color jitter, grayscale. Blur, then return as tensor.

In [5]:
simclr_transform = T.Compose([
    T.RandomResizedCrop(32),
    T.RandomHorizontalFlip(),
    T.RandomApply([T.ColorJitter(0.4,0.4,0.4,0.1)], p=0.8),
    T.RandomGrayscale(p=0.2),
    T.GaussianBlur(kernel_size=3),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])

# Dataset
Create two views (xi, xj) of the given image from the dataset.

In [6]:
class SimCLRDataset(Dataset):
    def __init__(self, base_dataset, transform):
        self.dataset = base_dataset
        self.transform = transform

    def __getitem__(self, index):
        image, _ = self.dataset[index]
        xi = self.transform(image)
        xj = self.transform(image)
        return xi, xj

    def __len__(self):
        return len(self.dataset)

# SimCLR Model
- Encoder: ResNet18, ResNet50, ResNet101
- Projection Head: maps features from encoder to 128-Dimensional Contrastive Space
- Output: contrastive loss $z$

In [7]:
class SimCLRModel(nn.Module):
    def __init__(self, model_type, projection_dim=128):
        super().__init__()
        base_model = model_type(weights=None)
        num_ftrs = base_model.fc.in_features
        base_model.fc = nn.Identity()
        self.encoder = base_model
        self.projection_head = nn.Sequential(
            nn.Linear(num_ftrs, 2048),
            nn.ReLU(),
            nn.Linear(2048, projection_dim)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projection_head(h)
        return z

# Loss Function
NT-Xent

In [8]:
def nt_xent_loss(z_i, z_j, temperature=0.05):
    z = torch.cat([z_i, z_j], dim=0)
    z = F.normalize(z, dim=1)

    similarity = torch.matmul(z, z.T)
    N = z_i.shape[0]

    mask = (~torch.eye(2*N, dtype=bool)).to(z.device)
    sim = similarity / temperature
    exp_sim = torch.exp(sim) * mask

    positive_sim = torch.exp(F.cosine_similarity(z_i, z_j) / temperature)
    positives = torch.cat([positive_sim, positive_sim], dim=0)

    denominator = exp_sim.sum(dim=1)
    loss = -torch.log(positives / denominator)
    return loss.mean()

ProxyNCA

In [9]:
def binarize_and_smooth_labels(T, nb_classes, smoothing_const=0.1):
    T = T.cpu().numpy()
    T = sklearn.preprocessing.label_binarize(T, classes=range(nb_classes))
    T = T * (1 - smoothing_const)
    T[T == 0] = smoothing_const / (nb_classes - 1)
    return torch.FloatTensor(T).to(device)


class ProxyNCA(nn.Module):
    def __init__(self, num_classes, embedding_dim=512, smoothing_const=0.1,
                 scaling_x=1.0, scaling_p=3.0):
        super().__init__()
        self.proxies = nn.Parameter(torch.randn(num_classes, embedding_dim) / 8)
        self.smoothing_const = smoothing_const
        self.scaling_x = scaling_x
        self.scaling_p = scaling_p
        self.num_classes = num_classes

    def forward(self, z, labels):
        P = F.normalize(self.proxies, p=2, dim=-1) * self.scaling_p
        Z = F.normalize(z, p=2, dim=-1) * self.scaling_x

        D = torch.cdist(Z, P) ** 2

        T = binarize_and_smooth_labels(labels, self.num_classes, self.smoothing_const)
        loss = torch.sum(-T * F.log_softmax(-D, dim=-1), dim=-1)
        return loss.mean()

# Training Loop

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_dataset = Cub2011(root='./cub2011', train=True, download=True)
contrastive_dataset = SimCLRDataset(train_dataset, simclr_transform)
train_loader = DataLoader(contrastive_dataset, batch_size=1024, shuffle=True, num_workers=2)
num_epochs = 200

for model_type in [(models.resnet18, "resnet18"), (models.resnet50, "resnet50"), (models.resnet101, "resnet101")]:
#for model_type in [(models.resnet50, "resnet50")]:
  model = SimCLRModel(model_type[0]).to(device)

  #load model from checkpoint
  #model.load_state_dict(torch.load("/content/test_model_resnet18.pth", map_location=device))

  optimizer = torch.optim.Adam(model.parameters(), lr=4e-4)

  for epoch in range(num_epochs):
      model.train()
      total_loss = 0
      for x_i, x_j in tqdm(train_loader):
          x_i, x_j = x_i.to(device), x_j.to(device)
          z_i = model(x_i)
          z_j = model(x_j)

          loss = nt_xent_loss(z_i, z_j)

          optimizer.zero_grad()
          loss.backward()
          optimizer.step()
          total_loss += loss.item()

      print(f"Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")

  torch.save(model.state_dict(), "test_model_" + model_type[1] + "_200ep.pth")

100%|██████████████████████████████████████████████████████████| 1.15G/1.15G [00:50<00:00, 23.0MB/s]
100%|█████████████████████████████████████████████████████████████████| 6/6 [00:05<00:00,  1.01it/s]


Epoch 1 | Loss: 7.4298


100%|█████████████████████████████████████████████████████████████████| 6/6 [00:05<00:00,  1.02it/s]


Epoch 2 | Loss: 7.1216


100%|█████████████████████████████████████████████████████████████████| 6/6 [00:05<00:00,  1.03it/s]


...

Epoch 45 | Loss: 6.6196


100%|█████████████████████████████████████████████████████████████████| 6/6 [00:06<00:00,  1.05s/it]


Epoch 46 | Loss: 6.6545


 33%|█████████████████████▋                                           | 2/6 [00:03<00:07,  1.75s/it]


KeyboardInterrupt: 

In [ ]:
#training loop for proxynca

#transform images for proxy
proxy_transform = T.Compose([
    T.Resize(64),
    T.RandomCrop(32),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
proxy_train_dataset = Cub2011(root='./cub2011', train=True, download=True, transform=proxy_transform)
proxy_train_loader = DataLoader(proxy_train_dataset, batch_size=128, shuffle=True, num_workers=2)


for model_type in [(models.resnet18, "resnet18"), (models.resnet50, "resnet50"), (models.resnet101, "resnet101")]:
    base = model_type[0](weights=None)
    embedding_dim = base.fc.in_features
    base.fc = nn.Identity()
    encoder = base.to(device)

    proxy_loss_fn = ProxyNCA(
        num_classes=100, #split training and tesing
        embedding_dim=embedding_dim,
    ).to(device)

    optimizer = torch.optim.Adam(
        list(encoder.parameters()) + list(proxy_loss_fn.parameters()),
        lr=1e-4,
        weight_decay=1e-4
    )

    for epoch in range(5):
        encoder.train()
        proxy_loss_fn.train()
        total_loss = 0
        for images, labels in tqdm(proxy_train_loader):
            images, labels = images.to(device), labels.to(device)
            z = encoder(images)

            loss = proxy_loss_fn(z, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"epoch {epoch+1} | Loss: {total_loss / len(proxy_train_loader):.4f}")

    torch.save({'encoder': encoder.state_dict(),'proxies': proxy_loss_fn.state_dict()}, "/content/proxynca_model_" + model_type[1] + ".pth")

Files already downloaded and verified


100%|██████████| 47/47 [00:12<00:00,  3.68it/s]


epoch 1 | Loss: 2.5516


100%|██████████| 47/47 [00:11<00:00,  4.09it/s]


epoch 2 | Loss: 2.5277


...

epoch 3 | Loss: 2.5338


100%|██████████| 47/47 [00:12<00:00,  3.86it/s]


epoch 4 | Loss: 2.5302


100%|██████████| 47/47 [00:12<00:00,  3.70it/s]


epoch 5 | Loss: 2.5322


### Evaluation - NT-XENT

In [12]:
#NT-Xent Eval | Resnet
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimCLRModel(models.resnet18, projection_dim=128).to(device)
model.load_state_dict(torch.load("test_model_resnet50_200ep.pth", map_location=device))

#similar transformation t o training transformation except without the random variations
evaluation_transform = T.Compose([
    T.Resize((32, 32)),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))])

#load dataset
test_dataset = Cub2011(root='./cub2011', train=False, download=True, transform = evaluation_transform)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)
print(model)
model.eval()

RuntimeError: Error(s) in loading state_dict for SimCLRModel:
	Unexpected key(s) in state_dict: "encoder.layer1.2.conv1.weight", "encoder.layer1.2.bn1.weight", "encoder.layer1.2.bn1.bias", "encoder.layer1.2.bn1.running_mean", "encoder.layer1.2.bn1.running_var", "encoder.layer1.2.bn1.num_batches_tracked", "encoder.layer1.2.conv2.weight", "encoder.layer1.2.bn2.weight", "encoder.layer1.2.bn2.bias", "encoder.layer1.2.bn2.running_mean", "encoder.layer1.2.bn2.running_var", "encoder.layer1.2.bn2.num_batches_tracked", "encoder.layer1.2.conv3.weight", "encoder.layer1.2.bn3.weight", "encoder.layer1.2.bn3.bias", "encoder.layer1.2.bn3.running_mean", "encoder.layer1.2.bn3.running_var", "encoder.layer1.2.bn3.num_batches_tracked", "encoder.layer1.0.conv3.weight", "encoder.layer1.0.bn3.weight", "encoder.layer1.0.bn3.bias", "encoder.layer1.0.bn3.running_mean", "encoder.layer1.0.bn3.running_var", "encoder.layer1.0.bn3.num_batches_tracked", "encoder.layer1.0.downsample.0.weight", "encoder.layer1.0.downsa

In [ ]:
#function to extract the image embeddings of entire test dataset, creating the evaluation vector space
def extract_embeddings_labels(model, loader):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    embeddings, labels = [], []
    #taken from g4g
    with torch.no_grad():
      for x, y in loader:
            x = x.to(device)
            h = model.encoder(x)
            embeddings.append(h.cpu())
            labels.append(y)
    return torch.cat(embeddings), torch.cat(labels)

In [ ]:
test_embeddings, test_labels = extract_embeddings_labels(model, test_loader)
print(test_embeddings)

correct = 0
for i in range(len(test_labels)):
  #get current embedding
  current_embedding = test_embeddings[i]
  #calculate distance from current embedding to all embeddings
  #https://discuss.pytorch.org/t/k-nearest-neighbor-in-pytorch/59695
  #euclidean distance
  dist = torch.norm(test_embeddings - current_embedding, dim=1, p=None)
  #exclude current image from k_nn calculation
  dist[i] = float('inf')

  #get k nearest neighbors
  k_nearest = dist.topk(3, largest=False)

  #check if any of the nearest neighbors have the same label as the current test image
  for j in range(len(k_nearest.indices)):
    if (test_labels[k_nearest.indices[j]] == test_labels[i]):
      correct += 1

print(correct / len(test_labels))

Proxy-NCA Eval

In [ ]:
evaluation_transform_NCA = T.Compose([
    T.Resize((64, 64)),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))])

def calc_recall_at_k(T, Y, k):
    """
    T : [nb_samples] (target labels)
    Y : [nb_samples x k] (k predicted labels/neighbours)
    """
    s = sum([1 for t, y in zip(T, Y) if t in y[:k]])
    return s / (1. * len(T))

def assign_by_euclidian_at_k(X, T, k):
    """
    X : [nb_samples x nb_features], e.g. 100 x 64 (embeddings)
    k : for each sample, assign target labels of k nearest points
    """
    distances = torch.cdist(X, X)
    # get nearest points
    indices = distances.topk(k + 1, largest=False)[1][:, 1: k + 1]
    return np.array([[T[i] for i in ii] for ii in indices])

def cluster_by_kmeans(X, nb_clusters):
    """
    xs : embeddings with shape [nb_samples, nb_features]
    nb_clusters : in this case, must be equal to number of classes
    """
    return sklearn.cluster.KMeans(nb_clusters).fit(X).labels_

def calc_normalized_mutual_information(ys, xs_clustered):
    return sklearn.metrics.cluster.normalized_mutual_info_score(xs_clustered, ys)

def predict_batchwise(model, dataloader):
    # list with N lists, where N = |{image, label, index}|
    model_is_training = model.training
    model.eval()
    ds = dataloader.dataset
    A = [[] for i in range(len(ds[0]))]
    with torch.no_grad():

        # extract batches (A becomes list of samples)
        for batch in dataloader:
            for i, J in enumerate(batch):
                # i = 0: sz_batch * images
                # i = 1: sz_batch * labels
                # i = 2: sz_batch * indices
                if i == 0:
                    # move images to device of model (approximate device)
                    J = J.to(list(model.parameters())[0].device)
                    # predict model output for image
                    J = model(J).cpu()
                for j in J:
                    A[i].append(j)
    model.train()
    model.train(model_is_training) # revert to previous training state
    return [torch.stack(A[i]) for i in range(len(A))]

def evaluate(model, dataloader, with_nmi = True):
    nb_classes = dataloader.dataset.nb_classes()

    # calculate embeddings with model and get targets
    X, T, *_ = predict_batchwise(model, dataloader)

    if with_nmi:
        # calculate NMI with kmeans clustering
        nmi = calc_normalized_mutual_information(
            T,
            cluster_by_kmeans(
                X, nb_classes
            )
        )
        print("NMI: {:.3f}".format(nmi * 100))

    # get predictions by assigning nearest 8 neighbors with euclidian
    Y = assign_by_euclidian_at_k(X, T, 8)
    Y = torch.from_numpy(Y)

    # calculate recall @ 1, 2, 4, 8
    recall = []
    for k in [1, 2, 4, 8]:
        r_at_k = calc_recall_at_k(T, Y, k)
        recall.append(r_at_k)
        print("R@{} : {:.3f}".format(k, 100 * r_at_k))
    if with_nmi:
        return recall, nmi
    else:
        return recall

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize the base encoder model (ResNet50 without the final classification layer)
base_model = models.resnet50(weights=None)
embedding_dim = base_model.fc.in_features
base_model.fc = nn.Identity()
encoder_model = base_model.to(device)

# Initialize the ProxyNCA loss function (num_classes from training, embedding_dim from resnet50)
proxy_nca_loss_fn = ProxyNCA(num_classes=100, embedding_dim=embedding_dim).to(device)

# Load the saved state dictionaries for both encoder and proxies
checkpoint = torch.load("/content/proxynca_model_resnet50.pth", map_location=device)
encoder_model.load_state_dict(checkpoint['encoder'])
proxy_nca_loss_fn.load_state_dict(checkpoint['proxies'])

# Set models to evaluation mode
encoder_model.eval()
proxy_nca_loss_fn.eval()

# Load dataset for evaluation
test_dataset = Cub2011(root='./cub2011', train=False, download=True, transform=evaluation_transform_NCA)
# Fix: Add nb_classes method to the dataset instance
test_dataset.nb_classes = lambda: 200 # Cub2011 dataset has 200 classes
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

with torch.no_grad():
    logging.info("**Evaluating...**")
    # Pass the encoder_model for evaluation, as it produces the embeddings
    evaluate(encoder_model, test_loader)


Files already downloaded and verified
NMI: 23.385
R@1 : 0.759
R@2 : 1.484
R@4 : 2.813
R@8 : 5.488


In [2]:
#NCA Eval | Resnet
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ProxyNCA(num_classes=100, embedding_dim=embedding_dim).to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/Colab/COMP597FinalProject/test_model_resnet50_200ep.pth", map_location=device))

#similar transformation t o training transformation except without the random variations
evaluation_transform_NCA = T.Compose([
    T.Resize((64, 64)),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))])

#load dataset
test_dataset = Cub2011(root='./cub2011', train=False, download=True, transform = evaluation_transform_NCA)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)
print(model)
model.eval()

NameError: name 'torch' is not defined

In [ ]:
#function to extract the image embeddings of entire test dataset, creating the evaluation vector space
def extract_embeddings_labels(model, loader):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    embeddings, labels = [], []
    #taken from g4g
    with torch.no_grad():
      for x, y in loader:
            x = x.to(device)
            h = model.encoder(x)
            embeddings.append(h.cpu())
            labels.append(y)
    return torch.cat(embeddings), torch.cat(labels)

In [ ]:
test_embeddings, test_labels = extract_embeddings_labels(model, test_loader)
print(test_embeddings)

correct = 0
for i in range(len(test_labels)):
  #get current embedding
  current_embedding = test_embeddings[i]
  #calculate distance from current embedding to all embeddings
  #https://discuss.pytorch.org/t/k-nearest-neighbor-in-pytorch/59695
  #euclidean distance
  dist = torch.norm(test_embeddings - current_embedding, dim=1, p=None)
  #exclude current image from k_nn calculation
  dist[i] = float('inf')

  #get k nearest neighbors
  k_nearest = dist.topk(3, largest=False)

  #check if any of the nearest neighbors have the same label as the current test image
  for j in range(len(k_nearest.indices)):
    if (test_labels[k_nearest.indices[j]] == test_labels[i]):
      correct += 1

print(correct / len(test_labels))